# UABR — Phase 3: pretrain the 40M base model on `data_v1` (1B tokens)
**Kaggle settings:** Accelerator **GPU T4 x2** (one GPU is used) · Internet **On** · **Add Input → Your Work → the Phase 2 notebook** (its `data_v1` output).

**Run it twice, same code path:**
1. `QUICK = True` → **Run All** (~10 min): 300 steps on the real data, a fact probe and text samples. Checks the data loads and nothing breaks.
2. `QUICK = False` → **Save Version › Save & Run All (Commit)**: the full 15,260-step run with `--compile` (~5–6 h at the 54k tokens/s measured in Phase 1). Close the browser; check the log later.

If the full run is ever cut short, add the stopped version's output as an input, set `RESUME` to its `runs/pretrain_40m` folder and commit again: it continues from the last checkpoint.

In [ ]:
QUICK = True
RESUME = ""   # e.g. "/kaggle/input/<this-notebook-name>/runs/pretrain_40m" to continue a cut-short run
REPO_URL = "https://github.com/Prajwal-13-dev/uabr_slm.git"

import os, glob, json, subprocess
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/uabr_slm"
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

found = glob.glob("/kaggle/input/**/data_v1/manifest.json", recursive=True)
assert found, "data_v1 not found: Add Input -> Your Work -> the Phase 2 notebook"
DATA = os.path.dirname(found[0])
m = json.load(open(f"{DATA}/manifest.json"))
audit = json.load(open(f"{DATA}/audit_report.json"))
tokrep = json.load(open(f"{DATA}/tokenizer/tokenizer_report.json"))
print("DATA       :", DATA)
print("train      :", f"{m['pretrain']['train_tokens']:,} tokens | reached target: {m['pretrain']['reached_target']}")
print("vocab      :", tokrep["vocab_size"], "| vocab source:", m["vocab_source"])
print("audits     :", "ALL PASS" if audit["all_pass"] else audit["pass"])
assert tokrep["vocab_size"] <= 12000, "tokenizer is bigger than the model's vocab_size"
if not QUICK:
    assert m["pretrain"]["reached_target"], "data_v1 is the QUICK build - rebuild Phase 2 with QUICK = False"
    assert audit["all_pass"], "fix the failing audit before pretraining"
RUN = f"{WORK}/runs/quick_real" if QUICK else f"{WORK}/runs/pretrain_40m"
!git log --oneline -1
!python scripts/check_gpu.py | head -4

## 1. Train
Logs every 10 steps; validation and a checkpoint every 500 steps (every 100 in QUICK). Watch that `loss_scale` stays stable and `grad_norm` stays finite.

In [ ]:
if QUICK:
    !rm -rf {RUN}
    !python src/train/pretrain.py --config configs/pretrain.yaml --data_dir {DATA}/pretrain --output_dir {RUN} --max_steps 300 --save_every 100 --log_every 20 --compile
else:
    resume = f"--resume {RESUME}" if RESUME else ""
    !python src/train/pretrain.py --config configs/pretrain.yaml --data_dir {DATA}/pretrain --output_dir {RUN} --max_minutes 690 --compile {resume}

## 2. Loss curve

In [ ]:
import matplotlib.pyplot as plt
rows = [json.loads(l) for l in open(f"{RUN}/metrics.jsonl")]
tr = [r for r in rows if "loss" in r]; va = [r for r in rows if "val_loss" in r]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([r["tokens_seen"] / 1e6 for r in tr], [r["loss"] for r in tr], lw=0.8, label="train")
ax.plot([r["step"] * 65536 / 1e6 for r in va], [r["val_loss"] for r in va], "o-", label="validation")
ax.set_xlabel("tokens (millions)"); ax.set_ylabel("loss"); ax.legend(); ax.grid(alpha=0.3)
plt.savefig(f"{RUN}/loss_curve.png", dpi=120, bbox_inches="tight"); plt.show()
print("last train:", tr[-1]); print("last val  :", va[-1] if va else None)

## 3. Fact probe (did it store the controlled facts?) + text samples
Expect exact match and gold log-prob to rise with exposure tier: 80 > 20 > 5 > withheld (0). The QUICK run will show nothing yet; that is normal.

In [ ]:
!python scripts/probe_facts.py --ckpt {RUN} --data {DATA} --per_tier 300
!python scripts/sample_text.py --ckpt {RUN} --data {DATA}

## 4. Freeze `base_v1` (full run only)
Exports weights + config with the data and tokenizer hashes. Every later phase starts from this file.

In [ ]:
man = json.load(open(f"{RUN}/manifest.json"))
print("status:", man.get("status"), "| sessions:", [(s["start_step"], s.get("end_step")) for s in man["sessions"]])
if not QUICK and man.get("status") == "completed":
    import sys; sys.path.insert(0, REPO_DIR)
    from src.model.load import export_weights
    out = export_weights(RUN, f"{WORK}/base_v1.pt", {
        "name": "base_v1", "data_version": m["data_version"], "tokenizer_sha256": m["tokenizer_sha256"],
        "git_commit": man.get("git_commit")})
    print("saved", out, f"{os.path.getsize(out) / 1e6:.0f} MB")
    !cp {DATA}/tokenizer/tokenizer.json {WORK}/tokenizer.json
elif not QUICK:
    print("Run not finished: commit again with RESUME set (see top).")
!du -sh {WORK}/runs/* 2>/dev/null

## 5. Gate (record in the sheet)
- Validation loss fell smoothly, with no NaN.
- The samples are mostly fluent English.
- The fact probe passes: recall rises with exposure tier.
- `base_v1.pt` exists in this version's output.

If the 80-statement tier is still near 0 at the end, do not start Phase 4. See *Risks* in the plan (continued pretraining with more statements).